
# 🛒 Buy Again Recommender — Person 1

**Responsibility:** Buy Again
**Algorithm:** Recency + Frequency + Quantity scoring
**Output:** Per-customer ranked list of crops/listings the customer is likely to repurchase, saved as a `.pkl` model.

**Pipeline**
1. Load data (`customer_order`, `order_item`, `seller_listing`, `crop`)
2. Build a unified purchase dataset
3. Feature engineering (purchase_count, total_quantity, last_purchase_date, days_since_last_purchase, purchase_frequency)
4. Compute a Buy Again score (Recency + Frequency + Quantity)
5. Generate ranked recommendations per customer
6. Evaluate with metrics + plots
7. Save everything to a `.pkl` file
8. Test with `get_buy_again(customer_id=101)`

> ⚠️ **Before running:** upload your Excel file (with sheets for `customer_order`, `order_item`, `seller_listing`, `crop`) and edit the `CONFIG` cell below so the column names match your file exactly.


## 1. Setup — install & import libraries

In [ ]:

!pip install -q pandas numpy scikit-learn joblib openpyxl matplotlib


In [ ]:

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)


## 2. Upload the Excel dataset

Run the cell below in Colab and select your `.xlsx` file. If you'd rather point to a file already
sitting in your Drive/runtime, just set `DATA_PATH` directly and skip the upload widget.

In [ ]:

# --- Option A: interactive upload (Google Colab) ---------------------------
try:
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
    print(f"Loaded file: {DATA_PATH}")
except ImportError:
    # --- Option B: not running in Colab -> set the path manually -----------
    DATA_PATH = "dataset.xlsx"   # <-- change this if needed
    print("Not running in Colab. Using DATA_PATH =", DATA_PATH)


## 3. CONFIG — sheet & column names

Edit this cell so the sheet names and column names match your Excel file exactly.
`STANDARD_NAME: "your_actual_column_name"` — the pipeline renames everything to the
standard names on the left, so nothing else below needs to change.

In [ ]:

CONFIG = {
    "sheet_names": {
        "customer_order": "customer_order",
        "order_item":     "order_item",
        "seller_listing": "seller_listing",
        "crop":           "crop",
    },

    # customer_order sheet
    "customer_order_cols": {
        "order_id":     "order_id",
        "customer_id":  "customer_id",
        "order_status": "order_status",
        "created_at":   "created_at",
    },

    # order_item sheet
    "order_item_cols": {
        "order_id":   "order_id",
        "listing_id": "listing_id",
        "quantity":   "quantity",
    },

    # seller_listing sheet
    "seller_listing_cols": {
        "listing_id": "listing_id",
        "crop_id":    "crop_id",
    },

    # crop sheet
    "crop_cols": {
        "crop_id":   "crop_id",
        "crop_name": "crop_name",
    },

    # order statuses that count as a valid completed purchase
    # set to None to keep every status
    "valid_order_statuses": ["completed", "delivered", "fulfilled"],

    # scoring weights (must sum to 1.0)
    "weights": {
        "recency":   0.40,
        "frequency": 0.35,
        "quantity":  0.25,
    },

    # exponential decay constant (days) for the recency score
    "recency_decay_days": 30,

    # minimum number of purchases before an item is eligible to be "bought again"
    "min_purchase_count": 1,

    # how many recommendations to keep per customer
    "top_n": 10,
}

assert abs(sum(CONFIG["weights"].values()) - 1.0) < 1e-6, "weights must sum to 1.0"
print("Config loaded.")


## 4. Load data

In [ ]:

xls = pd.ExcelFile(DATA_PATH)
print("Sheets found in workbook:", xls.sheet_names)

customer_order = pd.read_excel(xls, CONFIG["sheet_names"]["customer_order"])
order_item     = pd.read_excel(xls, CONFIG["sheet_names"]["order_item"])
seller_listing = pd.read_excel(xls, CONFIG["sheet_names"]["seller_listing"])
crop           = pd.read_excel(xls, CONFIG["sheet_names"]["crop"])

for name, df in [("customer_order", customer_order), ("order_item", order_item),
                  ("seller_listing", seller_listing), ("crop", crop)]:
    print(f"\n{name}: {df.shape}")
    print(df.columns.tolist())


In [ ]:

# Standardize column names using CONFIG so the rest of the notebook never
# has to worry about the raw Excel headers again.
def rename_cols(df, col_map):
    inv_map = {v: k for k, v in col_map.items()}
    return df.rename(columns=inv_map)[list(col_map.keys())]

customer_order = rename_cols(customer_order, CONFIG["customer_order_cols"])
order_item     = rename_cols(order_item, CONFIG["order_item_cols"])
seller_listing = rename_cols(seller_listing, CONFIG["seller_listing_cols"])
crop           = rename_cols(crop, CONFIG["crop_cols"])

customer_order["created_at"] = pd.to_datetime(customer_order["created_at"], errors="coerce")

display(customer_order.head())
display(order_item.head())
display(seller_listing.head())
display(crop.head())


## 5. Build the purchase dataset

In [ ]:

purchase_df = (
    order_item
    .merge(customer_order, on="order_id", how="inner")
    .merge(seller_listing, on="listing_id", how="inner")
    .merge(crop, on="crop_id", how="inner")
)

# Keep only valid/completed orders if configured
if CONFIG["valid_order_statuses"]:
    before = len(purchase_df)
    purchase_df = purchase_df[
        purchase_df["order_status"].astype(str).str.lower().isin(
            [s.lower() for s in CONFIG["valid_order_statuses"]]
        )
    ]
    print(f"Filtered order_status {CONFIG['valid_order_statuses']}: {before} -> {len(purchase_df)} rows")

purchase_df = purchase_df.dropna(subset=["customer_id", "listing_id", "created_at", "quantity"])

purchase_df = purchase_df[[
    "customer_id", "order_id", "listing_id", "crop_id", "crop_name",
    "quantity", "created_at", "order_status",
]].reset_index(drop=True)

print("purchase_df shape:", purchase_df.shape)
purchase_df.head()


## 6. Feature engineering

In [ ]:

reference_date = purchase_df["created_at"].max() + pd.Timedelta(days=1)
print("Reference date used for recency calculations:", reference_date)

grouped = (
    purchase_df
    .groupby(["customer_id", "listing_id", "crop_id", "crop_name"])
    .agg(
        purchase_count=("order_id", "nunique"),
        total_quantity=("quantity", "sum"),
        last_purchase_date=("created_at", "max"),
        first_purchase_date=("created_at", "min"),
    )
    .reset_index()
)

grouped["days_since_last_purchase"] = (reference_date - grouped["last_purchase_date"]).dt.days

# customer tenure = days between a customer's first ever purchase (any item) and reference date
customer_span = (
    purchase_df.groupby("customer_id")["created_at"]
    .agg(customer_first_purchase="min", customer_last_purchase="max")
    .reset_index()
)
customer_span["tenure_days"] = (
    reference_date - customer_span["customer_first_purchase"]
).dt.days.clip(lower=1)

grouped = grouped.merge(customer_span[["customer_id", "tenure_days"]], on="customer_id", how="left")
grouped["purchase_frequency"] = grouped["purchase_count"] / grouped["tenure_days"]

grouped = grouped[grouped["purchase_count"] >= CONFIG["min_purchase_count"]].reset_index(drop=True)

print("Feature table shape:", grouped.shape)
grouped.head(10)


## 7. Buy Again score — Recency + Frequency + Quantity

In [ ]:

def min_max_within_customer(df, col, new_col):
    \"\"\"Scale a column to [0,1] within each customer's own purchase history.\"\"\"
    def _scale(s):
        rng = s.max() - s.min()
        if rng == 0:
            return pd.Series(1.0, index=s.index)  # only one distinct value -> full score
        return (s - s.min()) / rng
    df[new_col] = df.groupby("customer_id")[col].transform(_scale)
    return df

# Recency: exponential decay, naturally bounded in (0, 1]
decay = CONFIG["recency_decay_days"]
grouped["recency_score"] = np.exp(-grouped["days_since_last_purchase"] / decay)

# Frequency & Quantity: scaled relative to the customer's own purchase history
grouped = min_max_within_customer(grouped, "purchase_count", "frequency_score")
grouped = min_max_within_customer(grouped, "total_quantity", "quantity_score")

w = CONFIG["weights"]
grouped["buy_again_score"] = (
    w["recency"]   * grouped["recency_score"] +
    w["frequency"] * grouped["frequency_score"] +
    w["quantity"]  * grouped["quantity_score"]
).round(4)

grouped.sort_values(["customer_id", "buy_again_score"], ascending=[True, False]).head(10)


## 8. Generate recommendations

In [ ]:

def build_reason(row):
    return (
        f"Bought {int(row['purchase_count'])}x, "
        f"last purchased {int(row['days_since_last_purchase'])} day(s) ago, "
        f"total quantity {row['total_quantity']:.0f}"
    )

grouped["reason"] = grouped.apply(build_reason, axis=1)
grouped["recommendation_type"] = "BUY_AGAIN"

recommendations = (
    grouped
    .sort_values(["customer_id", "buy_again_score"], ascending=[True, False])
    .groupby("customer_id")
    .head(CONFIG["top_n"])
    .reset_index(drop=True)
)

recommendations["rank"] = recommendations.groupby("customer_id")["buy_again_score"].rank(
    method="first", ascending=False
).astype(int)

final_recommendations = recommendations[[
    "customer_id", "listing_id", "crop_id", "crop_name",
    "buy_again_score", "recommendation_type", "reason", "rank",
]].rename(columns={"buy_again_score": "score"})

print("Final recommendations shape:", final_recommendations.shape)
final_recommendations.head(15)


## 9. Metrics & evaluation

In [ ]:

total_customers_in_data = customer_order["customer_id"].nunique()
customers_with_recs = final_recommendations["customer_id"].nunique()
coverage_pct = 100 * customers_with_recs / total_customers_in_data

print("===== BUY AGAIN — MODEL METRICS =====")
print(f"Total customers (all orders):      {total_customers_in_data}")
print(f"Customers with recommendations:    {customers_with_recs}")
print(f"Coverage:                          {coverage_pct:.2f}%")
print(f"Total recommendation rows:         {len(final_recommendations)}")
print(f"Avg recommendations / customer:    {len(final_recommendations)/customers_with_recs:.2f}")
print()
print("Score distribution:")
display(final_recommendations["score"].describe())


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(final_recommendations["score"], bins=20, color="#4C8BF5", edgecolor="white")
axes[0].set_title("Distribution of Buy Again Scores")
axes[0].set_xlabel("score")
axes[0].set_ylabel("count")

top_crops = (
    final_recommendations["crop_name"]
    .value_counts()
    .head(10)
    .sort_values()
)
axes[1].barh(top_crops.index, top_crops.values, color="#34A853")
axes[1].set_title("Top 10 Most-Recommended Crops")
axes[1].set_xlabel("times recommended")

plt.tight_layout()
plt.show()


## 10. Wrap it in a reusable model class

In [ ]:

class BuyAgainRecommender:
    \"\"\"Lightweight wrapper around the precomputed recommendation table.\"\"\"

    def __init__(self, recommendations: pd.DataFrame, config: dict, metrics: dict):
        self.recommendations = recommendations.reset_index(drop=True)
        self.config = config
        self.metrics = metrics
        self.trained_at = datetime.now()

    def get_buy_again(self, customer_id, top_n=5):
        recs = (
            self.recommendations[self.recommendations["customer_id"] == customer_id]
            .sort_values("score", ascending=False)
            .head(top_n)
            .reset_index(drop=True)
        )
        if recs.empty:
            print(f"No purchase history found for customer_id={customer_id}")
        return recs[["crop_name", "listing_id", "crop_id", "score", "reason"]]

metrics_summary = {
    "total_customers_in_data": int(total_customers_in_data),
    "customers_with_recommendations": int(customers_with_recs),
    "coverage_pct": round(coverage_pct, 2),
    "total_recommendation_rows": int(len(final_recommendations)),
    "avg_recommendations_per_customer": round(len(final_recommendations)/customers_with_recs, 2),
    "score_mean": round(float(final_recommendations["score"].mean()), 4),
    "score_std": round(float(final_recommendations["score"].std()), 4),
    "score_min": round(float(final_recommendations["score"].min()), 4),
    "score_max": round(float(final_recommendations["score"].max()), 4),
}

model = BuyAgainRecommender(final_recommendations, CONFIG, metrics_summary)
print("Model created.")
metrics_summary


## 11. Test — `get_buy_again(customer_id=101)`

Expected style of output:
```
Tomato       0.94
Potato       0.89
Onion        0.85
```

In [ ]:

def get_buy_again(customer_id, top_n=5):
    return model.get_buy_again(customer_id, top_n=top_n)

# pick a real customer_id from the data to demo with (swap in 101 if that exists in your file)
sample_customer_id = final_recommendations["customer_id"].iloc[0]
print(f"Demo for customer_id={sample_customer_id}\n")

result = get_buy_again(customer_id=sample_customer_id, top_n=5)
for _, row in result.iterrows():
    print(f"{row['crop_name']:<15} {row['score']:.2f}")

result


In [ ]:

# Try the exact call requested in the spec — will print "no purchase history" if 101 isn't in your data
get_buy_again(customer_id=101)


## 12. Save the final `.pkl` artifact

In [ ]:

import os

OUTPUT_PATH = "buy_again_model.pkl"

artifact = {
    "model": model,                        # BuyAgainRecommender instance (has .get_buy_again)
    "recommendations": final_recommendations,
    "feature_table": grouped,
    "config": CONFIG,
    "metrics": metrics_summary,
    "generated_at": datetime.now().isoformat(),
}

joblib.dump(artifact, OUTPUT_PATH)
print(f"Saved model artifact -> {os.path.abspath(OUTPUT_PATH)}")
print(f"File size: {os.path.getsize(OUTPUT_PATH) / 1024:.1f} KB")


In [ ]:

# Sanity check: reload from disk and re-run the test call
loaded = joblib.load(OUTPUT_PATH)
loaded_model = loaded["model"]
print("Reloaded metrics:", loaded["metrics"])
print()
loaded_model.get_buy_again(customer_id=sample_customer_id, top_n=5)


In [ ]:

# Optional: download the .pkl locally if running in Colab
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except ImportError:
    print("Not in Colab — file saved at:", os.path.abspath(OUTPUT_PATH))


## 13. Also export a CSV copy of the recommendations (optional)

In [ ]:

final_recommendations.to_csv("buy_again_recommendations.csv", index=False)
print("Also saved buy_again_recommendations.csv")
